# Finance AI

**Finance AI Agent – Data Analysis, Preprocessing, Regression Models, Evaluation and Model Saving**

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")

## Load Finance Dataset

In [ ]:
finance_df = pd.read_csv("finance_ml_df.csv")

print(finance_df.shape)
finance_df.head()

## Dataset Information

In [ ]:
finance_df.info()

## Check Missing Values

In [ ]:
finance_df.isnull().sum()

## Check Duplicate Rows

In [ ]:
finance_df.duplicated().sum()

In [ ]:
finance_df = finance_df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", finance_df.shape)

## Descriptive Statistics

In [ ]:
finance_df.describe().T

## Correlation Heatmap

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(finance_df.corr(numeric_only=True), cmap="coolwarm", annot=False)
plt.title("Finance Dataset Correlation Heatmap")
plt.show()

## Target Columns

In [ ]:
target_columns = [
    "target_next_week_revenue",
    "target_next_week_net_profit"
]

target_columns

## Feature Columns

In [ ]:
feature_columns = [
    col for col in finance_df.columns
    if col not in target_columns
]

feature_columns

## Separate Features and Targets

In [ ]:
X = finance_df[feature_columns].copy()

y_revenue = finance_df["target_next_week_revenue"].copy()
y_profit = finance_df["target_next_week_net_profit"].copy()

print("X shape:", X.shape)
print("Revenue target shape:", y_revenue.shape)
print("Profit target shape:", y_profit.shape)

## Convert Columns to Numeric

In [ ]:
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

y_revenue = pd.to_numeric(y_revenue, errors="coerce")
y_profit = pd.to_numeric(y_profit, errors="coerce")

print(X.dtypes)

## Handle Missing Target Values

In [ ]:
valid_rows = y_revenue.notna() & y_profit.notna()

X = X.loc[valid_rows].reset_index(drop=True)
y_revenue = y_revenue.loc[valid_rows].reset_index(drop=True)
y_profit = y_profit.loc[valid_rows].reset_index(drop=True)

print("Final dataset size:", len(X))

## Train-Test Split

In [ ]:
# Keep the same row order for a time-based finance dataset.
split_index = int(len(X) * 0.80)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_revenue_train = y_revenue.iloc[:split_index].copy()
y_revenue_test = y_revenue.iloc[split_index:].copy()

y_profit_train = y_profit.iloc[:split_index].copy()
y_profit_test = y_profit.iloc[split_index:].copy()

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## Preprocessing

In [ ]:
preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

## Linear Regression Model

In [ ]:
linear_revenue_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

linear_profit_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

In [ ]:
linear_revenue_model.fit(X_train, y_revenue_train)
linear_profit_model.fit(X_train, y_profit_train)

print("Linear Regression models trained successfully.")

## Random Forest Regression Model

In [ ]:
rf_revenue_model = Pipeline([
    ("preprocessor", SimpleImputer(strategy="median")),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        min_samples_leaf=2
    ))
])

rf_profit_model = Pipeline([
    ("preprocessor", SimpleImputer(strategy="median")),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        min_samples_leaf=2
    ))
])

In [ ]:
rf_revenue_model.fit(X_train, y_revenue_train)
rf_profit_model.fit(X_train, y_profit_train)

print("Random Forest models trained successfully.")

## Gradient Boosting Regression Model

In [ ]:
gb_revenue_model = Pipeline([
    ("preprocessor", SimpleImputer(strategy="median")),
    ("regressor", GradientBoostingRegressor(random_state=42))
])

gb_profit_model = Pipeline([
    ("preprocessor", SimpleImputer(strategy="median")),
    ("regressor", GradientBoostingRegressor(random_state=42))
])

In [ ]:
gb_revenue_model.fit(X_train, y_revenue_train)
gb_profit_model.fit(X_train, y_profit_train)

print("Gradient Boosting models trained successfully.")

## Revenue Predictions

In [ ]:
revenue_predictions = {
    "Linear Regression": linear_revenue_model.predict(X_test),
    "Random Forest": rf_revenue_model.predict(X_test),
    "Gradient Boosting": gb_revenue_model.predict(X_test)
}

for name, pred in revenue_predictions.items():
    print(name, ":", pred[:5])

## Net Profit Predictions

In [ ]:
profit_predictions = {
    "Linear Regression": linear_profit_model.predict(X_test),
    "Random Forest": rf_profit_model.predict(X_test),
    "Gradient Boosting": gb_profit_model.predict(X_test)
}

for name, pred in profit_predictions.items():
    print(name, ":", pred[:5])

## Evaluation Function

In [ ]:
def evaluate_regression(y_true, y_pred, model_name, target_name):
    mse = mean_squared_error(y_true, y_pred)

    return {
        "Target": target_name,
        "Model": model_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_true, y_pred)
    }

## Revenue Model Evaluation

In [ ]:
revenue_results = []

for name, pred in revenue_predictions.items():
    revenue_results.append(
        evaluate_regression(
            y_revenue_test,
            pred,
            name,
            "Next Week Revenue"
        )
    )

revenue_results_df = pd.DataFrame(revenue_results)
revenue_results_df

## Net Profit Model Evaluation

In [ ]:
profit_results = []

for name, pred in profit_predictions.items():
    profit_results.append(
        evaluate_regression(
            y_profit_test,
            pred,
            name,
            "Next Week Net Profit"
        )
    )

profit_results_df = pd.DataFrame(profit_results)
profit_results_df

## Combined Model Evaluation

In [ ]:
model_evaluation = pd.concat(
    [revenue_results_df, profit_results_df],
    ignore_index=True
)

model_evaluation

## Actual vs Predicted Revenue

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(y_revenue_test.to_numpy(), label="Actual")
plt.plot(
    revenue_predictions["Random Forest"],
    label="Random Forest Predicted"
)
plt.title("Actual vs Predicted Next Week Revenue")
plt.xlabel("Test Row")
plt.ylabel("Revenue")
plt.legend()
plt.show()

## Actual vs Predicted Net Profit

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(y_profit_test.to_numpy(), label="Actual")
plt.plot(
    profit_predictions["Random Forest"],
    label="Random Forest Predicted"
)
plt.title("Actual vs Predicted Next Week Net Profit")
plt.xlabel("Test Row")
plt.ylabel("Net Profit")
plt.legend()
plt.show()

## Select Random Forest for Finance Agent

In [ ]:
# Random Forest is used as the deployed model in finance_agent.py.
# The evaluation table above is retained so the models can be compared.

final_revenue_model = rf_revenue_model
final_profit_model = rf_profit_model

print("Final Finance Agent models selected.")

## Save Finance Models

In [ ]:
model_dir = "models"
os.makedirs(model_dir, exist_ok=True)

joblib.dump(
    final_revenue_model,
    os.path.join(model_dir, "target_next_week_revenue_model.pkl")
)

joblib.dump(
    final_profit_model,
    os.path.join(model_dir, "target_next_week_net_profit_model.pkl")
)

joblib.dump(
    feature_columns,
    os.path.join(model_dir, "finance_features.pkl")
)

model_evaluation.to_csv(
    os.path.join(model_dir, "model_evaluation.csv"),
    index=False
)

print("Saved:")
print("1. target_next_week_revenue_model.pkl")
print("2. target_next_week_net_profit_model.pkl")
print("3. finance_features.pkl")
print("4. model_evaluation.csv")

## Test Saved Models

In [ ]:
saved_revenue_model = joblib.load(
    os.path.join(model_dir, "target_next_week_revenue_model.pkl")
)

saved_profit_model = joblib.load(
    os.path.join(model_dir, "target_next_week_net_profit_model.pkl")
)

sample = X_test.iloc[[0]]

print(
    "Predicted next week revenue:",
    saved_revenue_model.predict(sample)[0]
)

print(
    "Predicted next week net profit:",
    saved_profit_model.predict(sample)[0]
)

## Finance AI Training Completed

The trained models and feature list are now ready to be used by `finance_agent.py`.